# Public Data, Capacity, and Cloud Integration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/CST4714_OER_Rebuild/notebooks/06_public_data_capacity_integration.ipynb)

This notebook uses a small teaching snapshot of the U.S. Cybersecurity and
Infrastructure Security Agency's Known Exploited Vulnerabilities catalog. It
connects four skills: evaluating a source, checking data quality, reasoning about
distribution, and loading records without creating duplicates.

**By the end, you will be able to:**

- identify source, retrieval, transformation, and use limits before importing;
- check nulls, duplicates, identifiers, and dates;
- compare cardinality, frequency, monotonicity, and query targeting;
- explain range and hashed distribution with measured evidence;
- load records idempotently into SQLite, Atlas, or PostgreSQL; and
- verify more than a successful connection or insert count.

The default path is fully offline and requires no account. Cloud paths are
optional and prompt for credentials only at runtime.

## Resource and License Boundary

The notebook prose and code are course OER. The CISA records come from an official
U.S. government feed and are not represented as original course data. The
embedded snapshot preserves source metadata and a description of the field
selection. It is compact classroom evidence, not a current vulnerability-
management source.

Official feed:
<https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json>

In [1]:
# SQLite is built into Python. PyMongo and Psycopg support optional cloud paths.
%pip -q install pymongo "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from bisect import bisect_right
from collections import Counter
from datetime import date
from getpass import getpass
from hashlib import sha256
import json
import sqlite3
import urllib.request

from pymongo import MongoClient
import psycopg

print("Notebook libraries are ready.")

Notebook libraries are ready.


## 1. Choose the Versioned Snapshot or Current Feed

`USE_LIVE_FEED` is `False` by default. That makes the class result reproducible
and keeps the notebook usable during an outage. Change it to `True` only when you
intend to inspect the current official feed. Current results will differ from the
versioned teaching snapshot.

In [3]:
OFFLINE_SNAPSHOT = json.loads(r'''{"title": "CISA Catalog of Known Exploited Vulnerabilities", "catalogVersion": "2026.07.10", "sourceDateReleased": "2026-07-10T17:00:25.7327Z", "retrievedAt": "2026-07-13T06:19:55.069283+00:00", "sourceUrl": "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json", "selection": "first 75 source records; documented fields only", "count": 75, "vulnerabilities": [{"cveID": "CVE-2026-56291", "vendorProject": "Balbooa", "product": "Forms", "vulnerabilityName": "Balbooa Forms Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-10", "dueDate": "2026-07-13", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-48939", "vendorProject": "iCagenda", "product": "iCagenda", "vulnerabilityName": "iCagenda Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-10", "dueDate": "2026-07-13", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-48908", "vendorProject": "JoomShaper", "product": "SP Page Builder", "vulnerabilityName": "JoomShaper SP Page Builder Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-55255", "vendorProject": "Langflow", "product": "Langflow", "vulnerabilityName": "Langflow Authorization Bypass Through User-Controlled Key Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-639"]}, {"cveID": "CVE-2026-56290", "vendorProject": "Joomlack", "product": "Page Builder", "vulnerabilityName": "Joomlack Page Builder Improper Access Control Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-48282", "vendorProject": "Adobe", "product": "ColdFusion", "vulnerabilityName": "Adobe ColdFusion Path Traversal Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-45659", "vendorProject": "Microsoft", "product": "SharePoint Server", "vulnerabilityName": "Microsoft SharePoint Server Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-07-01", "dueDate": "2026-07-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-502"]}, {"cveID": "CVE-2026-48558", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Authentication Bypass Vulnerability", "dateAdded": "2026-06-29", "dueDate": "2026-07-02", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-347"]}, {"cveID": "CVE-2026-12569", "vendorProject": "PTC", "product": "Windchill and FlexPLM", "vulnerabilityName": "PTC Windchill and FlexPLM Improper Input Validation Vulnerability", "dateAdded": "2026-06-25", "dueDate": "2026-06-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20", "CWE-502"]}, {"cveID": "CVE-2026-20230", "vendorProject": "Cisco", "product": "Unified Communications Manager", "vulnerabilityName": "Cisco Unified Communications Manager Server-Side Request Forgery (SSRF) Vulnerability", "dateAdded": "2026-06-25", "dueDate": "2026-06-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-918"]}, {"cveID": "CVE-2025-67038", "vendorProject": "Lantronix", "product": "EDS5000", "vulnerabilityName": "Lantronix EDS5000 Code Injection Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78", "CWE-94"]}, {"cveID": "CVE-2026-34910", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Improper Input Validation Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2026-34909", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Path Traversal Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-34908", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Improper Access Control Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-20253", "vendorProject": "Splunk", "product": "Enterprise", "vulnerabilityName": "Splunk Enterprise Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-06-18", "dueDate": "2026-06-21", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-48907", "vendorProject": "Widget Factory", "product": "Joomla Content Editor ", "vulnerabilityName": "Widget Factory Joomla Content Editor Improper Access Control Vulnerability", "dateAdded": "2026-06-16", "dueDate": "2026-06-19", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-54420", "vendorProject": "LiteSpeed", "product": "cPanel Plugin", "vulnerabilityName": "LiteSpeed cPanel Plugin UNIX Symbolic Link (Symlink) Following Vulnerability", "dateAdded": "2026-06-15", "dueDate": "2026-06-18", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-61"]}, {"cveID": "CVE-2026-20262", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Directory or Path Traversal Vulnerability", "dateAdded": "2026-06-15", "dueDate": "2026-06-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-35273", "vendorProject": "Oracle", "product": " PeopleSoft Enterprise PeopleTools", "vulnerabilityName": "Oracle PeopleSoft Enterprise PeopleTools Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-06-12", "dueDate": "2026-06-15", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-10520", "vendorProject": "Ivanti", "product": "Sentry", "vulnerabilityName": "Ivanti Sentry OS Command Injection Vulnerability", "dateAdded": "2026-06-11", "dueDate": "2026-06-14", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78"]}, {"cveID": "CVE-2026-11645", "vendorProject": "Google", "product": "Chromium V8", "vulnerabilityName": "Google Chromium V8 Out-of-Bounds Read and Write Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-787", "CWE-125"]}, {"cveID": "CVE-2026-7473", "vendorProject": "Arista", "product": "Extensible Operating System", "vulnerabilityName": "Arista Extensible Operating System Incomplete Comparison with Missing Factors Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-1023"]}, {"cveID": "CVE-2026-20245", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Improper Encoding or Escaping of Output Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-116"]}, {"cveID": "CVE-2026-42271", "vendorProject": "BerriAI", "product": "LiteLLM", "vulnerabilityName": "BerriAI LiteLLM Command Injection Vulnerability", "dateAdded": "2026-06-08", "dueDate": "2026-06-22", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78", "CWE-77"]}, {"cveID": "CVE-2026-50751", "vendorProject": "Check Point", "product": "Security Gateway", "vulnerabilityName": "Check Point Security Gateway Improper Authentication Vulnerability", "dateAdded": "2026-06-08", "dueDate": "2026-06-11", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-287"]}, {"cveID": "CVE-2026-28318", "vendorProject": "SolarWinds", "product": "Serv-U", "vulnerabilityName": "SolarWinds Serv-U Uncontrolled Resource Consumption Vulnerability", "dateAdded": "2026-06-05", "dueDate": "2026-06-19", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-400"]}, {"cveID": "CVE-2026-45247", "vendorProject": "Mirasvit", "product": "Mirasvit Full Page Cache Warmer", "vulnerabilityName": "Mirasvit Full Page Cache Warmer Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-06-03", "dueDate": "2026-06-06", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-502"]}, {"cveID": "CVE-2022-0492", "vendorProject": "Linux", "product": "Kernel", "vulnerabilityName": "Linux Kernel Improper Authentication Vulnerability", "dateAdded": "2026-06-02", "dueDate": "2026-06-05", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287", "CWE-862"]}, {"cveID": "CVE-2025-48595", "vendorProject": "Android", "product": "Framework", "vulnerabilityName": "Android Framework Integer Overflow Vulnerability", "dateAdded": "2026-06-02", "dueDate": "2026-06-05", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-190"]}, {"cveID": "CVE-2024-21182", "vendorProject": "Oracle", "product": "WebLogic Server", "vulnerabilityName": "Oracle WebLogic Server Unspecified Vulnerability", "dateAdded": "2026-06-01", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2026-0257", "vendorProject": "Palo Alto Networks", "product": "PAN-OS", "vulnerabilityName": "Palo Alto Networks PAN-OS Authentication Bypass Vulnerability", "dateAdded": "2026-05-29", "dueDate": "2026-06-01", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-565"]}, {"cveID": "CVE-2026-48027", "vendorProject": "Nx", "product": "Nx Console", "vulnerabilityName": "Nx Console Embedded Malicious Code Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-06-10", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-506"]}, {"cveID": "CVE-2026-45321", "vendorProject": "TanStack", "product": "TanStack", "vulnerabilityName": "TanStack Unspecified Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-06-10", "knownRansomwareCampaignUse": "Known", "cwes": []}, {"cveID": "CVE-2026-8398", "vendorProject": "Daemon", "product": "Daemon Tools Lite", "vulnerabilityName": "Daemon Tools Lite Embedded Malicious Code Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-05-30", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-506"]}, {"cveID": "CVE-2026-48172", "vendorProject": "LiteSpeed", "product": "cPanel Plugin", "vulnerabilityName": "LiteSpeed cPanel Plugin Privilege Escalation Vulnerability", "dateAdded": "2026-05-26", "dueDate": "2026-05-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-266"]}, {"cveID": "CVE-2026-9082", "vendorProject": "Drupal", "product": "Core", "vulnerabilityName": "Drupal Core SQL Injection Vulnerability", "dateAdded": "2026-05-22", "dueDate": "2026-05-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-89"]}, {"cveID": "CVE-2025-34291", "vendorProject": "Langflow", "product": "Langflow", "vulnerabilityName": "Langflow Origin Validation Error Vulnerability", "dateAdded": "2026-05-21", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-346"]}, {"cveID": "CVE-2026-34926", "vendorProject": "Trend Micro", "product": "Apex One", "vulnerabilityName": "Trend Micro Apex One (On-Premise) Directory Traversal Vulnerability", "dateAdded": "2026-05-21", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-23"]}, {"cveID": "CVE-2008-4250", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Buffer Overflow Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-94"]}, {"cveID": "CVE-2009-1537", "vendorProject": "Microsoft", "product": "DirectX", "vulnerabilityName": "Microsoft DirectX NULL Byte Overwrite Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2009-3459", "vendorProject": "Adobe", "product": "Acrobat and Reader", "vulnerabilityName": "Adobe Acrobat and Reader Heap-Based Buffer Overflow Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-119"]}, {"cveID": "CVE-2010-0249", "vendorProject": "Microsoft", "product": "Internet Explorer", "vulnerabilityName": "Microsoft Internet Explorer Use-After-Free Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-416"]}, {"cveID": "CVE-2010-0806", "vendorProject": "Microsoft", "product": "Internet Explorer", "vulnerabilityName": "Microsoft Internet Explorer Use-After-Free Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-399"]}, {"cveID": "CVE-2026-41091", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Link Following Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-59"]}, {"cveID": "CVE-2026-45498", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Denial of Service Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2026-42897", "vendorProject": "Microsoft", "product": "Microsoft", "vulnerabilityName": "Microsoft Exchange Server Cross-Site Scripting Vulnerability", "dateAdded": "2026-05-15", "dueDate": "2026-05-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-79"]}, {"cveID": "CVE-2026-20182", "vendorProject": "Cisco", "product": "Catalyst SD-WAN", "vulnerabilityName": "Cisco Catalyst SD-WAN Controller Authentication Bypass Vulnerability", "dateAdded": "2026-05-14", "dueDate": "2026-05-17", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287"]}, {"cveID": "CVE-2026-42208", "vendorProject": "BerriAI", "product": "LiteLLM", "vulnerabilityName": "BerriAI LiteLLM SQL Injection Vulnerability", "dateAdded": "2026-05-08", "dueDate": "2026-05-11", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-89"]}, {"cveID": "CVE-2026-6973", "vendorProject": "Ivanti", "product": "Endpoint Manager Mobile (EPMM)", "vulnerabilityName": "Ivanti Endpoint Manager Mobile (EPMM) Improper Input Validation Vulnerability", "dateAdded": "2026-05-07", "dueDate": "2026-05-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2026-0300", "vendorProject": "Palo Alto Networks", "product": "PAN-OS", "vulnerabilityName": "Palo Alto Networks PAN-OS Out-of-bounds Write Vulnerability", "dateAdded": "2026-05-06", "dueDate": "2026-05-09", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-787"]}, {"cveID": "CVE-2026-31431", "vendorProject": "Linux", "product": "Kernel", "vulnerabilityName": "Linux Kernel Incorrect Resource Transfer Between Spheres Vulnerability", "dateAdded": "2026-05-01", "dueDate": "2026-05-15", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-669"]}, {"cveID": "CVE-2026-41940", "vendorProject": "WebPros", "product": "cPanel & WHM and WP2 (WordPress Squared)", "vulnerabilityName": "WebPros cPanel & WHM and WP2 (WordPress Squared) Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-04-30", "dueDate": "2026-05-03", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-306"]}, {"cveID": "CVE-2024-1708", "vendorProject": "ConnectWise", "product": "ScreenConnect", "vulnerabilityName": "ConnectWise ScreenConnect Path Traversal Vulnerability", "dateAdded": "2026-04-28", "dueDate": "2026-05-12", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-32202", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Protection Mechanism Failure Vulnerability", "dateAdded": "2026-04-28", "dueDate": "2026-05-12", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-693"]}, {"cveID": "CVE-2025-29635", "vendorProject": "D-Link", "product": "DIR-823X", "vulnerabilityName": "D-Link DIR-823X Command Injection Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-77"]}, {"cveID": "CVE-2024-7399", "vendorProject": "Samsung", "product": "MagicINFO 9 Server", "vulnerabilityName": "Samsung MagicINFO 9 Server Path Traversal Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22", "CWE-434"]}, {"cveID": "CVE-2024-57728", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Path Traversal Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-22"]}, {"cveID": "CVE-2024-57726", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Missing Authorization Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-862"]}, {"cveID": "CVE-2026-39987", "vendorProject": "Marimo", "product": "Marimo", "vulnerabilityName": "Marimo Remote Code Execution Vulnerability", "dateAdded": "2026-04-23", "dueDate": "2026-05-07", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-33825", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Insufficient Granularity of Access Control Vulnerability", "dateAdded": "2026-04-22", "dueDate": "2026-05-06", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-1220"]}, {"cveID": "CVE-2026-20122", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manger", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Incorrect Use of Privileged APIs Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-648"]}, {"cveID": "CVE-2026-20133", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Exposure of Sensitive Information to an Unauthorized Actor Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-200"]}, {"cveID": "CVE-2025-2749", "vendorProject": "Kentico", "product": "Kentico Xperience", "vulnerabilityName": "Kentico Xperience Path Traversal Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22", "CWE-434"]}, {"cveID": "CVE-2023-27351", "vendorProject": "PaperCut", "product": "NG/MF", "vulnerabilityName": "PaperCut NG/MF Improper Authentication Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-287"]}, {"cveID": "CVE-2025-48700", "vendorProject": "Synacor", "product": "Zimbra Collaboration Suite (ZCS)", "vulnerabilityName": "Synacor Zimbra Collaboration Suite (ZCS) Cross-site Scripting Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-79"]}, {"cveID": "CVE-2026-20128", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Storing Passwords in a Recoverable Format Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-257"]}, {"cveID": "CVE-2025-32975", "vendorProject": "Quest", "product": "KACE Systems Management Appliance (SMA)", "vulnerabilityName": "Quest KACE Systems Management Appliance (SMA) Improper Authentication Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287"]}, {"cveID": "CVE-2024-27199", "vendorProject": "JetBrains", "product": "TeamCity", "vulnerabilityName": "JetBrains TeamCity Relative Path Traversal Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-23"]}, {"cveID": "CVE-2026-34197", "vendorProject": "Apache", "product": "ActiveMQ", "vulnerabilityName": "Apache ActiveMQ Improper Input Validation Vulnerability", "dateAdded": "2026-04-16", "dueDate": "2026-04-30", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20", "CWE-94"]}, {"cveID": "CVE-2009-0238", "vendorProject": "Microsoft", "product": "Office", "vulnerabilityName": "Microsoft Office Remote Code Execution", "dateAdded": "2026-04-14", "dueDate": "2026-04-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-94"]}, {"cveID": "CVE-2026-32201", "vendorProject": "Microsoft", "product": "SharePoint Server", "vulnerabilityName": "Microsoft SharePoint Server Improper Input Validation Vulnerability", "dateAdded": "2026-04-14", "dueDate": "2026-04-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2012-1854", "vendorProject": "Microsoft", "product": "Visual Basic for Applications (VBA)", "vulnerabilityName": "Microsoft Visual Basic for Applications Insecure Library Loading Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-426"]}, {"cveID": "CVE-2025-60710", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Link Following Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-59"]}, {"cveID": "CVE-2023-21529", "vendorProject": "Microsoft", "product": "Exchange Server", "vulnerabilityName": "Microsoft Exchange Server Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-502"]}, {"cveID": "CVE-2023-36424", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Out-of-Bounds Read Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-125"]}]}''')

USE_LIVE_FEED = False
OFFICIAL_FEED = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

if USE_LIVE_FEED:
    with urllib.request.urlopen(OFFICIAL_FEED, timeout=30) as response:
        source_package = json.load(response)
    source_mode = "current official feed"
else:
    source_package = OFFLINE_SNAPSHOT
    source_mode = "embedded versioned teaching snapshot"

print("Source mode:", source_mode)
print("Catalog version:", source_package.get("catalogVersion") or source_package.get("sourceCatalogVersion"))
print("Source release time:", source_package.get("dateReleased") or source_package.get("sourceDateReleased"))
print("Retrieval time:", source_package.get("retrievedAt", "live request in this runtime"))
print("Source URL:", source_package.get("sourceUrl", OFFICIAL_FEED))

Source mode: embedded versioned teaching snapshot
Catalog version: 2026.07.10
Source release time: 2026-07-10T17:00:25.7327Z
Retrieval time: 2026-07-13T06:19:55.069283+00:00
Source URL: https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json


### Normalize Only the Fields Used by This Lesson

The live feed contains more fields than the teaching snapshot. We intentionally
select the same compact fields in both paths. This is a modeling decision, not a
claim that omitted fields are unimportant.

In [4]:
raw_records = source_package.get("vulnerabilities", [])

# Keep the lesson small even when the current feed is selected.
records = []
for raw in raw_records[:75]:
    records.append({
        "cveID": raw.get("cveID"),
        "vendorProject": raw.get("vendorProject"),
        "product": raw.get("product"),
        "vulnerabilityName": raw.get("vulnerabilityName"),
        "dateAdded": raw.get("dateAdded"),
        "dueDate": raw.get("dueDate"),
        "knownRansomwareCampaignUse": raw.get("knownRansomwareCampaignUse"),
        "cwes": raw.get("cwes") or [],
    })

print("Selected records:", len(records))
print("Selected fields:", list(records[0]))
print("Example record:\n", json.dumps(records[0], indent=2))

Selected records: 75
Selected fields: ['cveID', 'vendorProject', 'product', 'vulnerabilityName', 'dateAdded', 'dueDate', 'knownRansomwareCampaignUse', 'cwes']
Example record:
 {
  "cveID": "CVE-2026-56291",
  "vendorProject": "Balbooa",
  "product": "Forms",
  "vulnerabilityName": "Balbooa Forms Unrestricted Upload of File with Dangerous Type Vulnerability",
  "dateAdded": "2026-07-10",
  "dueDate": "2026-07-13",
  "knownRansomwareCampaignUse": "Unknown",
  "cwes": [
    "CWE-434"
  ]
}


## 2. Audit Before Loading

A successful JSON parse does not prove the records are suitable for a database.
We check missing values, duplicate identifiers, and ISO date values first.

In [5]:
fields = list(records[0])
null_counts = {
    field: sum(record.get(field) in (None, "") for record in records)
    for field in fields
}
cve_counts = Counter(record["cveID"] for record in records)
duplicate_ids = sorted(cve_id for cve_id, count in cve_counts.items() if count > 1)

invalid_dates = []
for record in records:
    for field in ("dateAdded", "dueDate"):
        try:
            date.fromisoformat(record[field])
        except (TypeError, ValueError):
            invalid_dates.append((record["cveID"], field, record[field]))

print("Null counts:", null_counts)
print("Duplicate CVE IDs:", duplicate_ids)
print("Invalid ISO dates:", invalid_dates)
assert records and not duplicate_ids and not invalid_dates

Null counts: {'cveID': 0, 'vendorProject': 0, 'product': 0, 'vulnerabilityName': 0, 'dateAdded': 0, 'dueDate': 0, 'knownRansomwareCampaignUse': 0, 'cwes': 0}
Duplicate CVE IDs: []
Invalid ISO dates: []


### Ask a Question Before Choosing a Database Shape

Our first question is: **Which vendors occur most often in this selected
snapshot?** This describes the sample, not all vulnerabilities and not a vendor's
security quality. The limited sample and source order matter.

In [6]:
vendor_counts = Counter(record["vendorProject"] for record in records)
print("Five most frequent vendors in this selected sample:")
for vendor, count in vendor_counts.most_common(5):
    print(f"  {vendor}: {count}")

Five most frequent vendors in this selected sample:
  Microsoft: 16
  Cisco: 7
  SimpleHelp : 3
  Ubiquiti: 3
  Langflow: 2


## 3. Measure Candidate Distribution Keys

We compare four candidates:

- `vendorProject` can target vendor questions but may be skewed;
- `dateAdded` supports time questions but may be monotonic;
- `cveID` is highly distinct but does not target vendor questions; and
- `(vendorProject, product)` can divide some vendor groups further.

High cardinality alone is not enough. A useful decision also considers frequency,
write order, and actual query shapes.

In [7]:
candidate_values = {
    "vendorProject": [record["vendorProject"] for record in records],
    "dateAdded": [record["dateAdded"] for record in records],
    "cveID": [record["cveID"] for record in records],
    "vendorProject + product": [
        (record["vendorProject"], record["product"]) for record in records
    ],
}

print(f"{'candidate':28} {'distinct':>8} {'ratio':>8} {'largest value share':>20}")
for name, values in candidate_values.items():
    frequencies = Counter(values)
    distinct = len(frequencies)
    largest_share = max(frequencies.values()) / len(values)
    print(f"{name:28} {distinct:8d} {distinct / len(values):8.2f} {largest_share:20.2%}")

dates_in_source_order = [record["dateAdded"] for record in records]
descending_date_order = all(
    left >= right for left, right in zip(dates_in_source_order, dates_in_source_order[1:])
)
print("\nDate-added values are monotonic descending in source order:", descending_date_order)
print("This is an input-order observation, not proof of production write order.")

candidate                    distinct    ratio  largest value share
vendorProject                      42     0.56               21.33%
dateAdded                          38     0.51               10.67%
cveID                              75     1.00                1.33%
vendorProject + product            56     0.75                5.33%

Date-added values are monotonic descending in source order: True
This is an input-order observation, not proof of production write order.


## 4. Simulate Range and Hashed Placement

This is not a sharded MongoDB deployment. It is a deterministic thought
experiment that makes two tradeoffs visible.

For the range simulation, the oldest 80 percent establishes three date boundaries
and the newest 20 percent acts like later writes. A monotonic time key tends to
place those later writes in the current high range. For the hash simulation,
SHA-256 maps CVE IDs into four teaching buckets. MongoDB uses its own hashing and
balancing behavior; these buckets only illustrate distribution.

In [8]:
chronological = sorted(records, key=lambda record: record["dateAdded"])
split_at = max(1, int(len(chronological) * 0.80))
historical = chronological[:split_at]
later_writes = chronological[split_at:]

historical_dates = sorted(record["dateAdded"] for record in historical)
range_boundaries = [
    historical_dates[int(len(historical_dates) * fraction)]
    for fraction in (0.25, 0.50, 0.75)
]
range_buckets = Counter(
    bisect_right(range_boundaries, record["dateAdded"])
    for record in later_writes
)
hash_buckets = Counter(
    int(sha256(record["cveID"].encode("utf-8")).hexdigest(), 16) % 4
    for record in later_writes
)

print("Historical date boundaries:", range_boundaries)
print("Later-write range buckets:", dict(sorted(range_buckets.items())))
print("Later-write teaching hash buckets:", dict(sorted(hash_buckets.items())))
print("Later records tested:", len(later_writes))
assert sum(range_buckets.values()) == len(later_writes)
assert sum(hash_buckets.values()) == len(later_writes)

Historical date boundaries: ['2026-04-22', '2026-05-20', '2026-06-01']
Later-write range buckets: {3: 15}
Later-write teaching hash buckets: {0: 3, 1: 3, 2: 6, 3: 3}
Later records tested: 15


### Record a Capacity Recommendation

Complete these statements before loading:

1. **Current scale:** This sample contains ___ records and is/is not large enough
   to justify sharding because ___.
2. **Candidate evidence:** ___ has ___ distinct values; its largest value holds
   ___ percent of the sample.
3. **Query targeting:** The main question filters/groups by ___, so ___ would or
   would not help route that question.
4. **Tradeoff:** Range distribution preserves ___ but risks ___; hashed
   distribution improves ___ but weakens ___.
5. **Decision:** Do not shard yet, or select ___ only under the stated future
   workload, because ___.

## 5. Load One Target Idempotently

The default is SQLite. It gives every student a complete database path without an
account. You may additionally enable Atlas or PostgreSQL. Cloud paths prompt for
the URI and never print it.

An **idempotent** load can be rerun without adding duplicate logical records. We
use `cveID` as the stable key and an upsert or conflict update on each target.

In [9]:
LOAD_SQLITE = True
LOAD_ATLAS = False
LOAD_POSTGRES = False

if LOAD_SQLITE:
    con = sqlite3.connect(":memory:")
    con.execute('''
        CREATE TABLE IF NOT EXISTS kev_sample (
            cve_id TEXT PRIMARY KEY,
            vendor_project TEXT NOT NULL,
            product TEXT NOT NULL,
            vulnerability_name TEXT NOT NULL,
            date_added TEXT NOT NULL,
            due_date TEXT NOT NULL,
            ransomware_use TEXT,
            cwes_json TEXT NOT NULL
        )
    ''')

    rows = [
        (
            record["cveID"], record["vendorProject"], record["product"],
            record["vulnerabilityName"], record["dateAdded"], record["dueDate"],
            record["knownRansomwareCampaignUse"], json.dumps(record["cwes"]),
        )
        for record in records
    ]
    con.executemany('''
        INSERT INTO kev_sample VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT (cve_id) DO UPDATE SET
            vendor_project = excluded.vendor_project,
            product = excluded.product,
            vulnerability_name = excluded.vulnerability_name,
            date_added = excluded.date_added,
            due_date = excluded.due_date,
            ransomware_use = excluded.ransomware_use,
            cwes_json = excluded.cwes_json
    ''', rows)
    con.commit()
    print("SQLite upsert completed.")

SQLite upsert completed.


### Optional Atlas Path

Before enabling this cell, create or open an Atlas Free project, create a database
user, and add only the temporary network access required for the runtime. Use the
current `mongodb+srv://` connection string. Do not add `tlsInsecure=True`; fix the
URI, network access, driver, DNS, or TLS cause instead.

In [10]:
if LOAD_ATLAS:
    atlas_uri = getpass("Atlas connection URI (hidden): ")
    atlas_client = MongoClient(atlas_uri, serverSelectionTimeoutMS=15000)
    atlas_client.admin.command("ping")

    atlas_collection = atlas_client["cst4714_public_data"]["kev_sample"]
    for record in records:
        atlas_collection.replace_one({"cveID": record["cveID"]}, record, upsert=True)

    print("Atlas upsert completed. Observed count:", atlas_collection.count_documents({}))
else:
    print("Atlas path skipped. Set LOAD_ATLAS = True only when you intend to connect.")

Atlas path skipped. Set LOAD_ATLAS = True only when you intend to connect.


### Optional Supabase/PostgreSQL Path

Copy the current PostgreSQL connection URL from your provider and enter it when
prompted. If a notebook network cannot reach the direct IPv6 endpoint, use the
provider's current IPv4-compatible session-pooler URL. Do not place the URL in a
Markdown or code cell.

In [11]:
if LOAD_POSTGRES:
    postgres_url = getpass("PostgreSQL connection URL (hidden): ")
    with psycopg.connect(postgres_url) as connection:
        with connection.cursor() as cursor:
            cursor.execute("CREATE SCHEMA IF NOT EXISTS cst4714_oer")
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS cst4714_oer.kev_sample (
                    cve_id text PRIMARY KEY,
                    vendor_project text NOT NULL,
                    product text NOT NULL,
                    vulnerability_name text NOT NULL,
                    date_added date NOT NULL,
                    due_date date NOT NULL,
                    ransomware_use text,
                    cwes jsonb NOT NULL
                )
            ''')
            cursor.executemany('''
                INSERT INTO cst4714_oer.kev_sample VALUES
                    (%s, %s, %s, %s, %s, %s, %s, %s::jsonb)
                ON CONFLICT (cve_id) DO UPDATE SET
                    vendor_project = excluded.vendor_project,
                    product = excluded.product,
                    vulnerability_name = excluded.vulnerability_name,
                    date_added = excluded.date_added,
                    due_date = excluded.due_date,
                    ransomware_use = excluded.ransomware_use,
                    cwes = excluded.cwes
            ''', rows)
        connection.commit()
    print("PostgreSQL upsert completed.")
else:
    print("PostgreSQL path skipped. Set LOAD_POSTGRES = True only when you intend to connect.")

PostgreSQL path skipped. Set LOAD_POSTGRES = True only when you intend to connect.


## 6. Verify the Import

We verify four things:

1. expected and observed counts;
2. one known stable identifier;
3. a grouped question tied to the reason for loading; and
4. rerun behavior through the primary key and upsert.

These checks still do not prove completeness of the full live catalog, correct
authorization, backup readiness, performance under load, or production fitness.

In [12]:
known_id = records[0]["cveID"]

if LOAD_SQLITE:
    observed_count = con.execute("SELECT count(*) FROM kev_sample").fetchone()[0]
    known_row = con.execute(
        "SELECT cve_id, vendor_project, product FROM kev_sample WHERE cve_id = ?",
        [known_id],
    ).fetchone()
    grouped = con.execute('''
        SELECT vendor_project, count(*) AS vulnerability_count
        FROM kev_sample
        GROUP BY vendor_project
        ORDER BY vulnerability_count DESC, vendor_project
        LIMIT 5
    ''').fetchall()

    print("Expected count:", len(records))
    print("Observed SQLite count:", observed_count)
    print("Known identifier check:", known_row)
    print("Grouped result:", grouped)
    assert observed_count == len(records) and known_row is not None

if LOAD_ATLAS:
    atlas_known = atlas_collection.find_one(
        {"cveID": known_id}, {"_id": 0, "cveID": 1, "vendorProject": 1, "product": 1}
    )
    atlas_grouped = list(atlas_collection.aggregate([
        {"$group": {"_id": "$vendorProject", "vulnerability_count": {"$sum": 1}}},
        {"$sort": {"vulnerability_count": -1, "_id": 1}},
        {"$limit": 5},
    ]))
    print("Atlas known identifier check:", atlas_known)
    print("Atlas grouped result:", atlas_grouped)
    assert atlas_collection.count_documents({}) == len(records) and atlas_known

if LOAD_POSTGRES:
    with psycopg.connect(postgres_url) as connection:
        with connection.cursor() as cursor:
            cursor.execute("SELECT count(*) FROM cst4714_oer.kev_sample")
            postgres_count = cursor.fetchone()[0]
            cursor.execute(
                "SELECT cve_id, vendor_project, product FROM cst4714_oer.kev_sample WHERE cve_id = %s",
                (known_id,),
            )
            postgres_known = cursor.fetchone()
            cursor.execute('''
                SELECT vendor_project, count(*) AS vulnerability_count
                FROM cst4714_oer.kev_sample
                GROUP BY vendor_project
                ORDER BY vulnerability_count DESC, vendor_project
                LIMIT 5
            ''')
            postgres_grouped = cursor.fetchall()
    print("PostgreSQL observed count:", postgres_count)
    print("PostgreSQL known identifier check:", postgres_known)
    print("PostgreSQL grouped result:", postgres_grouped)
    assert postgres_count == len(records) and postgres_known

Expected count: 75
Observed SQLite count: 75
Known identifier check: ('CVE-2026-56291', 'Balbooa', 'Forms')
Grouped result: [('Microsoft', 16), ('Cisco', 7), ('SimpleHelp ', 3), ('Ubiquiti', 3), ('Adobe', 2)]


## Submission Record

Complete this record in the notebook:

**Source and snapshot:** [official source, live or embedded mode, catalog version,
retrieval/release date, and why this is only a teaching subset]

**Quality evidence:** [record count, null result, duplicate result, and date result]

**Capacity evidence:** [one candidate's cardinality and largest-value share;
range/hash observation; recommendation and tradeoff]

**Target and idempotency:** [SQLite, Atlas, or PostgreSQL; stable key and upsert
behavior]

**Verification:** [expected/observed count, known identifier, grouped question,
and one thing these checks do not prove]

**Credential check:** I confirm no URI, password, token, or API key appears in
cell source or output. [replace with yes]

## Final-Project Transfer

This checkpoint does not add or redefine final-project deliverables. Use the
[canonical final project](../final_project.md).

Write four sentences:

1. My project could use a public or synthetic dataset about ___.
2. The source of truth should be ___ because ___.
3. My first scale trigger would be measured as ___.
4. Before scaling, I would verify ___ and improve ___ because ___.

If you used a cloud path, close the client and remove temporary broad network
access after class.

In [13]:
if LOAD_ATLAS:
    atlas_client.close()
    print("Closed the Atlas client. Review and narrow temporary network access.")
if LOAD_SQLITE:
    con.close()
    print("Closed the disposable SQLite database.")

Closed the disposable SQLite database.
